In [ ]:
import pandas as pd
from collections import defaultdict
from intervaltree import IntervalTree
import pybedtools
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

print("Libraries loaded successfully")

## 1. Load Transcript Sequences

In [ ]:
def load_transcripts(fasta_path):
    """Load transcript sequences from FASTA"""
    print(f"Loading transcripts from {fasta_path}...")
    transcripts = {}
    current_id = None
    current_seq = []
    
    with open(fasta_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if current_id:
                    transcripts[current_id] = ''.join(current_seq)
                current_id = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line.upper())
        
        if current_id:
            transcripts[current_id] = ''.join(current_seq)
    
    print(f"  Loaded {len(transcripts):,} transcripts")
    return transcripts

transcripts = load_transcripts('../transcripts.fa')

## 2. Load CDS Annotations

In [ ]:
def load_cds_annotations(gtf_path):
    """Load CDS annotations in transcript coordinates"""
    print(f"\nLoading CDS annotations from {gtf_path}...")
    print("This may take a few minutes...")
    
    gtf = pybedtools.BedTool(gtf_path)
    
    cds_info = {}
    transcript_info = {}
    exons_by_transcript = defaultdict(list)
    cds_by_transcript = defaultdict(list)
    
    # Load exons and CDS
    for feature in gtf:
        attrs = {}
        for attr in feature.fields[8].split(';'):
            attr = attr.strip()
            if '"' in attr:
                key, value = attr.split(' "', 1)
                attrs[key] = value.rstrip('"')
        
        transcript_id = attrs.get('transcript_id')
        if not transcript_id:
            continue
        
        tx_id_base = transcript_id.split('.')[0]
        
        if tx_id_base not in transcript_info:
            transcript_info[tx_id_base] = {
                'gene_name': attrs.get('gene_name', ''),
                'gene_id': attrs.get('gene_id', ''),
                'biotype': attrs.get('transcript_type', attrs.get('transcript_biotype', '')),
                'chrom': feature.chrom,
                'strand': feature.strand
            }
        
        if feature.fields[2] == 'exon':
            exons_by_transcript[tx_id_base].append({
                'start': feature.start,
                'end': feature.end
            })
        elif feature.fields[2] == 'CDS':
            cds_by_transcript[tx_id_base].append({
                'start': feature.start,
                'end': feature.end
            })
    
    # Convert CDS to transcript coordinates
    def genomic_to_transcript_coord(genomic_pos, exons, strand):
        """Convert genomic position to transcript coordinate"""
        if strand == '+':
            tx_pos = 0
            for exon in exons:
                if genomic_pos < exon['start']:
                    return None
                elif genomic_pos < exon['end']:
                    return tx_pos + (genomic_pos - exon['start'])
                tx_pos += (exon['end'] - exon['start'])
            return None
        else:
            tx_pos = 0
            for exon in reversed(exons):
                if genomic_pos > exon['end']:
                    return None
                elif genomic_pos >= exon['start']:
                    return tx_pos + (exon['end'] - genomic_pos)
                tx_pos += (exon['end'] - exon['start'])
            return None
    
    for tx_id in cds_by_transcript:
        if tx_id not in exons_by_transcript:
            continue
        
        exons = sorted(exons_by_transcript[tx_id], key=lambda x: x['start'])
        cds_regions = cds_by_transcript[tx_id]
        strand = transcript_info[tx_id]['strand']
        
        # Map to transcript coordinates
        tx_cds = []
        for cds in cds_regions:
            cds_start_tx = genomic_to_transcript_coord(cds['start'], exons, strand)
            cds_end_tx = genomic_to_transcript_coord(cds['end'] - 1, exons, strand)
            
            if cds_start_tx is not None and cds_end_tx is not None:
                if strand == '-':
                    cds_start_tx, cds_end_tx = cds_end_tx, cds_start_tx
                tx_cds.append({'start': cds_start_tx, 'end': cds_end_tx + 1})
        
        if tx_cds:
            cds_start = min(c['start'] for c in tx_cds)
            cds_end = max(c['end'] for c in tx_cds)
            cds_info[tx_id] = {'start': cds_start, 'end': cds_end}
    
    print(f"  Loaded CDS for {len(cds_info):,} transcripts")
    return cds_info, transcript_info

cds_info, transcript_info = load_cds_annotations('../annotation.gtf')

## 3. Analysis Functions

### 3A. Stop-Free Regions

In [ ]:
def find_stop_free_regions(sequence, min_length_codons=30):
    """Find regions without stop codons (single frame)"""
    stop_codons = {'TAA', 'TAG', 'TGA'}
    regions = []
    
    region_start = 0
    i = 0
    
    while i < len(sequence) - 2:
        codon = sequence[i:i+3]
        
        if codon in stop_codons:
            if i - region_start >= min_length_codons * 3:
                regions.append({
                    'start': region_start,
                    'end': i,
                    'length': (i - region_start) // 3
                })
            region_start = i + 3
        
        i += 3
    
    if len(sequence) - region_start >= min_length_codons * 3:
        regions.append({
            'start': region_start,
            'end': len(sequence),
            'length': (len(sequence) - region_start) // 3
        })
    
    return regions

print("Stop-free analysis function defined")

### 3B. ATG-to-STOP ORFs

In [ ]:
def find_atg_to_stop_orfs(sequence, min_length_codons=30):
    """Find ATG-initiated ORFs (single frame)"""
    stop_codons = {'TAA', 'TAG', 'TGA'}
    orfs = []
    
    i = 0
    while i < len(sequence) - 2:
        codon = sequence[i:i+3]
        
        if codon == 'ATG':
            # Found start, look for stop
            orf_start = i
            j = i + 3
            
            while j < len(sequence) - 2:
                stop_codon = sequence[j:j+3]
                if stop_codon in stop_codons:
                    # Found stop
                    orf_length = (j - orf_start) // 3
                    if orf_length >= min_length_codons:
                        orfs.append({
                            'start': orf_start,
                            'end': j + 3,
                            'length': orf_length
                        })
                    break
                j += 3
        
        i += 3
    
    return orfs

print("ATG-to-STOP ORF analysis function defined")

### 3D. Find Triple-Frame Overlaps

In [ ]:
def find_triple_frame_overlaps(regions_by_frame, min_overlap_codons=30):
    """Find regions where all 3 frames overlap"""
    min_overlap_nt = min_overlap_codons * 3
    
    # Build interval trees for frames 1 and 2
    tree1 = IntervalTree()
    tree2 = IntervalTree()
    
    for region in regions_by_frame[1]:
        tree1.addi(region['start'], region['end'], region)
    
    for region in regions_by_frame[2]:
        tree2.addi(region['start'], region['end'], region)
    
    triple_frame_regions = []
    
    # For each frame 0 region, find overlaps with frames 1 and 2
    for region0 in regions_by_frame[0]:
        overlaps_f1 = tree1.overlap(region0['start'], region0['end'])
        
        for iv1 in overlaps_f1:
            region1 = iv1.data
            
            # Calculate dual-frame overlap
            dual_start = max(region0['start'], region1['start'])
            dual_end = min(region0['end'], region1['end'])
            dual_length = dual_end - dual_start
            
            if dual_length >= min_overlap_nt:
                # Check for frame 2 overlap
                overlaps_f2 = tree2.overlap(dual_start, dual_end)
                
                for iv2 in overlaps_f2:
                    region2 = iv2.data
                    triple_start = max(dual_start, region2['start'])
                    triple_end = min(dual_end, region2['end'])
                    triple_length = triple_end - triple_start
                    
                    if triple_length >= min_overlap_nt:
                        triple_frame_regions.append({
                            'start': triple_start,
                            'end': triple_end,
                            'length': triple_length,
                            'length_codons': triple_length // 3,
                            'frame0_length': region0['length'],
                            'frame1_length': region1['length'],
                            'frame2_length': region2['length']
                        })
    
    return triple_frame_regions

print("Triple-frame overlap function defined")

## 4. Run Analysis 1: Stop-Free Regions

In [ ]:
print("\n" + "="*70)
print("ANALYSIS 1: STOP-FREE TRIPLE-FRAME REGIONS")
print("="*70)

stop_free_results = []
processed = 0

for tx_id, sequence in transcripts.items():
    processed += 1
    if processed % 10000 == 0:
        print(f"  Processed {processed:,} / {len(transcripts):,} transcripts...")
    
    if len(sequence) < 300:
        continue
    
    # Find stop-free regions in all frames
    regions_by_frame = {}
    for frame in [0, 1, 2]:
        frame_seq = sequence[frame:]
        regions = find_stop_free_regions(frame_seq, min_length_codons=30)
        regions_by_frame[frame] = [
            {'start': r['start'] + frame, 'end': r['end'] + frame, 'length': r['length']}
            for r in regions
        ]
    
    # Find triple-frame overlaps
    triple_regions = find_triple_frame_overlaps(regions_by_frame, min_overlap_codons=30)
    
    if not triple_regions:
        continue
    
    # Get metadata
    tx_id_base = tx_id.split('.')[0]
    info = transcript_info.get(tx_id_base, {})
    cds = cds_info.get(tx_id_base)
    
    # Store results
    for region in triple_regions:
        # Check CDS overlap
        has_cds_overlap = False
        cds_overlap_length = 0
        
        if cds:
            cds_overlap_start = max(region['start'], cds['start'])
            cds_overlap_end = min(region['end'], cds['end'])
            if cds_overlap_end > cds_overlap_start:
                has_cds_overlap = True
                cds_overlap_length = cds_overlap_end - cds_overlap_start
        
        stop_free_results.append({
            'transcript_id': tx_id,
            'gene_name': info.get('gene_name', ''),
            'gene_id': info.get('gene_id', ''),
            'biotype': info.get('biotype', ''),
            'region_start': region['start'],
            'region_end': region['end'],
            'region_length_codons': region['length_codons'],
            'has_cds_overlap': has_cds_overlap,
            'cds_overlap_codons': cds_overlap_length // 3 if cds_overlap_length > 0 else 0,
            'cds_start': cds['start'] if cds else None,
            'cds_end': cds['end'] if cds else None,
            'frame0_length': region['frame0_length'],
            'frame1_length': region['frame1_length'],
            'frame2_length': region['frame2_length']
        })

df_stop_free = pd.DataFrame(stop_free_results)
df_stop_free = df_stop_free.sort_values('region_length_codons', ascending=False).reset_index(drop=True)
df_stop_free['rank'] = range(1, len(df_stop_free) + 1)

print(f"\nFound {len(df_stop_free):,} triple-frame stop-free regions")
print(f"Unique genes: {df_stop_free['gene_name'].nunique():,}")
print(f"Median length: {df_stop_free['region_length_codons'].median():.0f} codons")
print(f"≥100 codons: {len(df_stop_free[df_stop_free['region_length_codons'] >= 100]):,}")

## 5. Run Analysis 2: ATG-to-STOP ORFs

In [ ]:
print("\n" + "="*70)
print("ANALYSIS 2: ATG-TO-STOP TRIPLE-FRAME ORFs")
print("="*70)

atg_stop_results = []
processed = 0

for tx_id, sequence in transcripts.items():
    processed += 1
    if processed % 10000 == 0:
        print(f"  Processed {processed:,} / {len(transcripts):,} transcripts...")
    
    if len(sequence) < 300:
        continue
    
    # Find ATG-to-STOP ORFs in all frames
    regions_by_frame = {}
    for frame in [0, 1, 2]:
        frame_seq = sequence[frame:]
        orfs = find_atg_to_stop_orfs(frame_seq, min_length_codons=30)
        regions_by_frame[frame] = [
            {'start': r['start'] + frame, 'end': r['end'] + frame, 'length': r['length']}
            for r in orfs
        ]
    
    # Find triple-frame overlaps
    triple_regions = find_triple_frame_overlaps(regions_by_frame, min_overlap_codons=30)
    
    if not triple_regions:
        continue
    
    # Get metadata
    tx_id_base = tx_id.split('.')[0]
    info = transcript_info.get(tx_id_base, {})
    cds = cds_info.get(tx_id_base)
    
    # Store results
    for region in triple_regions:
        # Check CDS overlap
        has_cds_overlap = False
        cds_overlap_length = 0
        
        if cds:
            cds_overlap_start = max(region['start'], cds['start'])
            cds_overlap_end = min(region['end'], cds['end'])
            if cds_overlap_end > cds_overlap_start:
                has_cds_overlap = True
                cds_overlap_length = cds_overlap_end - cds_overlap_start
        
        atg_stop_results.append({
            'transcript_id': tx_id,
            'gene_name': info.get('gene_name', ''),
            'gene_id': info.get('gene_id', ''),
            'biotype': info.get('biotype', ''),
            'region_start': region['start'],
            'region_end': region['end'],
            'region_length_codons': region['length_codons'],
            'has_cds_overlap': has_cds_overlap,
            'cds_overlap_codons': cds_overlap_length // 3 if cds_overlap_length > 0 else 0,
            'cds_start': cds['start'] if cds else None,
            'cds_end': cds['end'] if cds else None,
            'frame0_length': region['frame0_length'],
            'frame1_length': region['frame1_length'],
            'frame2_length': region['frame2_length']
        })

df_atg_stop = pd.DataFrame(atg_stop_results)
df_atg_stop = df_atg_stop.sort_values('region_length_codons', ascending=False).reset_index(drop=True)
df_atg_stop['rank'] = range(1, len(df_atg_stop) + 1)

print(f"\nFound {len(df_atg_stop):,} triple-frame ATG-to-STOP regions")
print(f"Unique genes: {df_atg_stop['gene_name'].nunique():,}")
print(f"Median length: {df_atg_stop['region_length_codons'].median():.0f} codons")
print(f"≥100 codons: {len(df_atg_stop[df_atg_stop['region_length_codons'] >= 100]):,}")

## 7. Find SRD5A1 in All Analyses

In [ ]:
print("\n" + "="*70)
print("SRD5A1 ANALYSIS")
print("="*70)

for analysis_name, df in [('Stop-Free', df_stop_free), ('ATG-to-STOP', df_atg_stop)]:
    print(f"\n{analysis_name}:")
    srd5a1 = df[df['gene_name'] == 'SRD5A1']
    
    if len(srd5a1) > 0:
        print(f"  ✅ Found {len(srd5a1)} regions")
        
        best = srd5a1.iloc[0]
        percentile = 100 * (1 - best['rank'] / len(df))
        print(f"\n  Best region:")
        print(f"    Rank: #{best['rank']:,} / {len(df):,} ({percentile:.1f}th percentile)")
        print(f"    Length: {best['region_length_codons']} codons")
        if best['cds_overlap_codons'] > 0:
            print(f"    CDS overlap: {best['cds_overlap_codons']} codons")
    else:
        print("  ❌ Not found")

# Save results


In [ ]:
output_dir = Path('../results/triple_frame')
output_dir.mkdir(parents=True, exist_ok=True)

df_stop_free.to_csv(output_dir / 'stop_free_triple_frame.tsv', sep='\t', index=False)
print(f"\nSaved: {output_dir / 'stop_free_triple_frame.tsv'}")

df_atg_stop.to_csv(output_dir / 'atg_stop_triple_frame.tsv', sep='\t', index=False)
print(f"Saved: {output_dir / 'atg_stop_triple_frame.tsv'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

datasets = [
    (df_stop_free, 'Stop-Free', 'steelblue'),
    (df_atg_stop, 'ATG→STOP', 'forestgreen')
]

for idx, (df, title, color) in enumerate(datasets):
    ax = axes[idx]
    
    lengths = df['region_length_codons']
    bins = np.arange(30, lengths.max() + 10, 10)
    
    ax.hist(lengths, bins=bins, color=color, edgecolor='black', linewidth=0.5, alpha=0.7)
    ax.set_xlabel('Region length (codons)', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_yscale('log')
    ax.set_title(f'{title}\n(n={len(df):,})', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add statistics
    median = lengths.median()
    long_count = (lengths >= 100).sum()
    
    stats_text = f'Median={median:.0f}\n≥100 cod={long_count:,}'
    ax.text(0.98, 0.97, stats_text, transform=ax.transAxes,
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
            fontsize=9, family='monospace')

plt.tight_layout()

fig_path = '../results/figures/triple_frame_comparison.png'
Path(fig_path).parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.savefig(fig_path.replace('.png', '.pdf'), bbox_inches='tight')
print(f"\nSaved figure: {fig_path}")
plt.show()

## Reverse Complement Control Analysis

As a control, analyze reverse complement of all transcripts to assess background triple-frame occurrence.

**Rationale**: Reverse complement sequences should lack biological constraint, providing a null distribution for comparison.

In [ ]:
def reverse_complement(seq):
    """Return reverse complement of DNA sequence"""
    complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', 'N': 'N'}
    return ''.join(complement.get(base, base) for base in reversed(seq))

# Create reverse complement transcripts
print("Creating reverse complement transcripts...")
revcomp_transcripts = {}
for tx_id, seq in transcripts.items():
    revcomp_transcripts[tx_id] = reverse_complement(seq)

print(f"  Created {len(revcomp_transcripts):,} reverse complement sequences")

### Reverse Complement Analysis 1: Stop-Free

In [ ]:
print("\n" + "="*70)
print("REVERSE COMPLEMENT: STOP-FREE TRIPLE-FRAME REGIONS")
print("="*70)

revcomp_stop_free_results = []
processed = 0

for tx_id, sequence in revcomp_transcripts.items():
    processed += 1
    if processed % 10000 == 0:
        print(f"  Processed {processed:,} / {len(revcomp_transcripts):,} transcripts...")
    
    if len(sequence) < 300:
        continue
    
    # Find stop-free regions in all frames
    regions_by_frame = {}
    for frame in [0, 1, 2]:
        frame_seq = sequence[frame:]
        regions = find_stop_free_regions(frame_seq, min_length_codons=30)
        regions_by_frame[frame] = [
            {'start': r['start'] + frame, 'end': r['end'] + frame, 'length': r['length']}
            for r in regions
        ]
    
    # Find triple-frame overlaps
    triple_regions = find_triple_frame_overlaps(regions_by_frame, min_overlap_codons=30)
    
    if not triple_regions:
        continue
    
    # Get metadata
    tx_id_base = tx_id.split('.')[0]
    info = transcript_info.get(tx_id_base, {})
    
    # Store results (no CDS info for revcomp)
    for region in triple_regions:
        revcomp_stop_free_results.append({
            'transcript_id': tx_id,
            'gene_name': info.get('gene_name', ''),
            'gene_id': info.get('gene_id', ''),
            'biotype': info.get('biotype', ''),
            'region_start': region['start'],
            'region_end': region['end'],
            'region_length_codons': region['length_codons'],
            'frame0_length': region['frame0_length'],
            'frame1_length': region['frame1_length'],
            'frame2_length': region['frame2_length']
        })

df_revcomp_stop_free = pd.DataFrame(revcomp_stop_free_results)
df_revcomp_stop_free = df_revcomp_stop_free.sort_values('region_length_codons', ascending=False).reset_index(drop=True)
df_revcomp_stop_free['rank'] = range(1, len(df_revcomp_stop_free) + 1)

print(f"\nFound {len(df_revcomp_stop_free):,} reverse complement stop-free regions")
print(f"Unique genes: {df_revcomp_stop_free['gene_name'].nunique():,}")
print(f"Median length: {df_revcomp_stop_free['region_length_codons'].median():.0f} codons")
print(f"≥100 codons: {len(df_revcomp_stop_free[df_revcomp_stop_free['region_length_codons'] >= 100]):,}")

### Reverse Complement Analysis 2: ATG-to-STOP

In [ ]:
print("\n" + "="*70)
print("REVERSE COMPLEMENT: ATG-TO-STOP TRIPLE-FRAME ORFs")
print("="*70)

revcomp_atg_stop_results = []
processed = 0

for tx_id, sequence in revcomp_transcripts.items():
    processed += 1
    if processed % 10000 == 0:
        print(f"  Processed {processed:,} / {len(revcomp_transcripts):,} transcripts...")
    
    if len(sequence) < 300:
        continue
    
    regions_by_frame = {}
    for frame in [0, 1, 2]:
        frame_seq = sequence[frame:]
        orfs = find_atg_to_stop_orfs(frame_seq, min_length_codons=30)
        regions_by_frame[frame] = [
            {'start': r['start'] + frame, 'end': r['end'] + frame, 'length': r['length']}
            for r in orfs
        ]
    
    triple_regions = find_triple_frame_overlaps(regions_by_frame, min_overlap_codons=30)
    
    if not triple_regions:
        continue
    
    tx_id_base = tx_id.split('.')[0]
    info = transcript_info.get(tx_id_base, {})
    
    for region in triple_regions:
        revcomp_atg_stop_results.append({
            'transcript_id': tx_id,
            'gene_name': info.get('gene_name', ''),
            'gene_id': info.get('gene_id', ''),
            'biotype': info.get('biotype', ''),
            'region_start': region['start'],
            'region_end': region['end'],
            'region_length_codons': region['length_codons'],
            'frame0_length': region['frame0_length'],
            'frame1_length': region['frame1_length'],
            'frame2_length': region['frame2_length']
        })

df_revcomp_atg_stop = pd.DataFrame(revcomp_atg_stop_results)
df_revcomp_atg_stop = df_revcomp_atg_stop.sort_values('region_length_codons', ascending=False).reset_index(drop=True)
df_revcomp_atg_stop['rank'] = range(1, len(df_revcomp_atg_stop) + 1)

print(f"\nFound {len(df_revcomp_atg_stop):,} reverse complement ATG-to-STOP regions")
print(f"Unique genes: {df_revcomp_atg_stop['gene_name'].nunique():,}")
print(f"Median length: {df_revcomp_atg_stop['region_length_codons'].median():.0f} codons")
print(f"≥100 codons: {len(df_revcomp_atg_stop[df_revcomp_atg_stop['region_length_codons'] >= 100]):,}")

# Save reverse complement results



In [ ]:
revcomp_output_dir = Path('../results/revcomp_triple_frame')
revcomp_output_dir.mkdir(parents=True, exist_ok=True)

df_revcomp_stop_free.to_csv(revcomp_output_dir / 'revcomp_stop_free_triple_frame.tsv', sep='\t', index=False)
print(f"\nSaved: {revcomp_output_dir / 'revcomp_stop_free_triple_frame.tsv'}")

df_revcomp_atg_stop.to_csv(revcomp_output_dir / 'revcomp_atg_stop_triple_frame.tsv', sep='\t', index=False)
print(f"Saved: {revcomp_output_dir / 'revcomp_atg_stop_triple_frame.tsv'}")